In [1]:
import pandas as pd
import numpy as np

# Ajusta la ruta según dónde guardaste el CSV
df = pd.read_csv("artifacts/blocks_features.csv", parse_dates=["start", "end"])

df.shape, df.columns

((5566, 12),
 Index(['start', 'end', 'n', 'duration_min', 'mean_rate', 'max_rate',
        'unique_tags', 'dominant_tag_share', 'unique_msgs',
        'dominant_msg_share', 'flood_candidate', 'flood_severity'],
       dtype='str'))

In [2]:
P99_RATE = 180   # p99 alarms/min
P999_RATE = 345  # p99.9 alarms/min

df["is_severe_rate"] = df["max_rate"] >= P999_RATE
df["is_extreme_rate"] = df["max_rate"] >= P99_RATE

df[["max_rate","is_extreme_rate","is_severe_rate"]].describe()

,max_rate
count,5566.000000
mean,29.088573
std,148.867860
min,1.000000
25%,3.000000
50%,9.000000
75%,21.000000
max,7734.000000


In [3]:
def assign_flood_type(row):
    ut = row["unique_tags"]
    um = row["unique_msgs"]
    dts = row["dominant_tag_share"]
    dms = row["dominant_msg_share"]
    mr = row["max_rate"]
    n = row["n"]

    # 1) Mensaje dominante transversal (muchos tags afectados)
    if (mr >= P99_RATE) and (dms >= 0.70) and (ut >= 20):
        return "MESSAGE_DOMINANT"

    # 2) Oscilación / chatter de un punto (tag domina el bloque)
    if (mr >= P99_RATE) and (dts >= 0.85):
        return "OSCILLATION_SINGLE_POINT"

    # 3) Repetición de pocos tags/mensajes (muy poca diversidad)
    if (mr >= P99_RATE) and (ut <= 10) and (dms >= 0.70):
        return "TAG_DOMINANT_REPETITION"

    # 4) Evento masivo con repetición (evento grande + mensaje dominante medio)
    if (mr >= P99_RATE) and (dms >= 0.50) and (ut >= 10):
        return "MASS_EVENT_WITH_REPETITION"

    # 5) No tipificado / no flood (o flood leve)
    return "OTHER_OR_NO_FLOOD"


df["flood_type"] = df.apply(assign_flood_type, axis=1)

df["flood_type"].value_counts()

flood_type
OTHER_OR_NO_FLOOD             5482
MESSAGE_DOMINANT                26
MASS_EVENT_WITH_REPETITION      25
TAG_DOMINANT_REPETITION         20
OSCILLATION_SINGLE_POINT        13
Name: count, dtype: int64

In [4]:
pd.crosstab(df["flood_type"], df["flood_severity"], margins=True)

flood_severity,medium,none,severe,All
flood_type,,,,
MASS_EVENT_WITH_REPETITION,15,8,2,25
MESSAGE_DOMINANT,22,0,4,26
OSCILLATION_SINGLE_POINT,12,0,1,13
OTHER_OR_NO_FLOOD,17,5465,0,5482
TAG_DOMINANT_REPETITION,14,0,6,20
All,80,5473,13,5566


In [5]:
def top_examples(df, flood_type, k=5):
    sub = df[df["flood_type"] == flood_type].copy()
    return sub.sort_values("max_rate", ascending=False).head(k)[
        ["start","end","n","duration_min","max_rate","unique_tags","dominant_tag_share","unique_msgs","dominant_msg_share","flood_severity","flood_candidate"]
    ]

for t in ["MESSAGE_DOMINANT","OSCILLATION_SINGLE_POINT","TAG_DOMINANT_REPETITION","MASS_EVENT_WITH_REPETITION"]:
    print("\n====", t, "====")
    display(top_examples(df, t, k=5))


==== MESSAGE_DOMINANT ====


,start,end,n,duration_min,max_rate,unique_tags,dominant_tag_share,unique_msgs,dominant_msg_share,flood_severity,flood_candidate
2822,2025-05-03 09:07:08,2025-05-03 09:59:14,12453,52.100000,7734,2615,0.070585,100,0.733317,severe,True
1765,2025-02-02 14:08:37,2025-02-02 19:28:44,44880,320.116667,672,27,0.478810,19,0.957620,severe,True
1457,2025-01-02 14:00:30,2025-01-02 19:14:18,27882,313.800000,456,26,0.490209,18,0.980310,severe,True
4080,2025-09-04 02:54:19,2025-09-04 06:39:38,17175,225.316667,378,20,0.306550,14,0.919301,severe,True
3839,2025-08-08 08:55:12,2025-08-08 12:36:36,7782,221.400000,330,80,0.602544,42,0.839244,medium,True



==== OSCILLATION_SINGLE_POINT ====


,start,end,n,duration_min,max_rate,unique_tags,dominant_tag_share,unique_msgs,dominant_msg_share,flood_severity,flood_candidate
651,2024-06-10 00:00:00,2024-06-10 10:33:08,59259,633.133333,432,77,0.975042,40,0.495267,severe,True
4156,2025-09-06 23:48:11,2025-09-07 20:50:20,61974,1262.150000,342,131,0.927050,49,0.470326,medium,True
652,2024-06-10 10:38:55,2024-06-11 01:05:58,64965,867.050000,315,132,0.908474,40,0.493558,medium,True
1145,2024-10-12 04:53:40,2024-10-12 12:21:32,24600,447.866667,291,110,0.911707,49,0.480976,medium,True
567,2024-05-10 00:00:00,2024-05-11 01:44:22,127794,1544.366667,288,114,0.917649,42,0.466289,medium,True



==== TAG_DOMINANT_REPETITION ====


,start,end,n,duration_min,max_rate,unique_tags,dominant_tag_share,unique_msgs,dominant_msg_share,flood_severity,flood_candidate
4138,2025-09-05 21:01:15,2025-09-05 21:20:53,3369,19.633333,576,6,0.325022,4,0.975067,severe,True
5352,2025-12-05 03:32:04,2025-12-05 05:53:15,17694,141.183333,540,9,0.335198,6,0.725331,severe,True
4477,2025-10-05 21:44:20,2025-10-05 21:54:53,2493,10.550000,486,3,0.333333,1,1.000000,severe,True
4498,2025-10-06 07:00:34,2025-10-06 07:25:59,5493,25.416667,468,6,0.332059,3,0.997815,severe,True
4499,2025-10-06 07:31:42,2025-10-06 08:26:18,7875,54.600000,378,9,0.331048,7,0.996952,severe,True



==== MASS_EVENT_WITH_REPETITION ====


,start,end,n,duration_min,max_rate,unique_tags,dominant_tag_share,unique_msgs,dominant_msg_share,flood_severity,flood_candidate
4379,2025-10-04 00:40:01,2025-10-04 01:59:18,11847,79.283333,735,16,0.302102,10,0.906305,severe,True
3072,2025-06-02 11:16:50,2025-06-03 07:09:34,56340,1192.733333,420,454,0.180777,113,0.653461,severe,True
1636,2025-01-07 09:22:29,2025-01-07 10:41:32,2220,79.050000,342,59,0.167568,26,0.509459,none,False
1646,2025-01-07 22:28:33,2025-01-07 23:04:22,1539,35.816667,288,17,0.259259,12,0.777778,medium,True
1712,2025-01-10 13:55:44,2025-01-10 18:59:34,14886,303.833333,270,105,0.310157,52,0.620314,medium,True


In [6]:
audit = (
    df[df["flood_type"].isin(["MESSAGE_DOMINANT","OSCILLATION_SINGLE_POINT","TAG_DOMINANT_REPETITION","MASS_EVENT_WITH_REPETITION"])]
    .sort_values(["flood_type","flood_severity","max_rate"], ascending=[True, False, False])
    .groupby("flood_type")
    .head(2)
    .reset_index(drop=True)
)

audit[["flood_type","start","end","max_rate","unique_tags","dominant_msg_share","dominant_tag_share","flood_severity"]]

,flood_type,start,end,max_rate,unique_tags,dominant_msg_share,dominant_tag_share,flood_severity
0,MASS_EVENT_WITH_REPETITION,2025-10-04 00:40:01,2025-10-04 01:59:18,735,16,0.906305,0.302102,severe
1,MASS_EVENT_WITH_REPETITION,2025-06-02 11:16:50,2025-06-03 07:09:34,420,454,0.653461,0.180777,severe
2,MESSAGE_DOMINANT,2025-05-03 09:07:08,2025-05-03 09:59:14,7734,2615,0.733317,0.070585,severe
3,MESSAGE_DOMINANT,2025-02-02 14:08:37,2025-02-02 19:28:44,672,27,0.957620,0.478810,severe
4,OSCILLATION_SINGLE_POINT,2024-06-10 00:00:00,2024-06-10 10:33:08,432,77,0.495267,0.975042,severe
5,OSCILLATION_SINGLE_POINT,2025-09-06 23:48:11,2025-09-07 20:50:20,342,131,0.470326,0.927050,medium
6,TAG_DOMINANT_REPETITION,2025-09-05 21:01:15,2025-09-05 21:20:53,576,6,0.975067,0.325022,severe
7,TAG_DOMINANT_REPETITION,2025-12-05 03:32:04,2025-12-05 05:53:15,540,9,0.725331,0.335198,severe


In [7]:
import os
import pandas as pd
import pyodbc

SERVER = "10.147.17.185"
PORT = 1433
USERNAME = "cmpcuser"
PASSWORD = os.getenv("OTMS_SQL_PASSWORD_DEV")
DRIVER = "{ODBC Driver 17 for SQL Server}"

DB = "cmpc_20240925_093000"
SCHEMA = "dbo"
TABLE = "ypf_alarms"

conn_str = (
    f"DRIVER={DRIVER};"
    f"SERVER={SERVER},{PORT};"
    f"DATABASE={DB};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str)
print("Conectado.")

Conectado.


In [8]:
start = "2025-05-03 09:07:08"
end   = "2025-05-03 09:59:14"

In [9]:
q = f"""
SELECT TOP 15
    ALARM_DESCRIPTION,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY ALARM_DESCRIPTION
ORDER BY cnt DESC
"""

df_msg = pd.read_sql(q, conn, params=[start, end])
df_msg["share"] = df_msg["cnt"] / df_msg["cnt"].sum()
df_msg

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\1763310739.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_msg = pd.read_sql(q, conn, params=[start, end])


,ALARM_DESCRIPTION,cnt,share
0,ALARM MESSAGES ENABLED,9132,0.770243
1,ALARMA,606,0.051113
2,NORMAL,582,0.049089
3,BAJA PRESION,576,0.048583
4,NaN,171,0.014423
5,OFF,165,0.013917
6,FALLA,123,0.010374
7,BAJO CAUDAL,90,0.007591
8,DETENIDO,78,0.006579
9,BAD I/O,72,0.006073


In [10]:
q = f"""
SELECT TOP 15
    TAG_DESCRIPTION,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY TAG_DESCRIPTION
ORDER BY cnt DESC
"""

df_tag = pd.read_sql(q, conn, params=[start, end])
df_tag["share"] = df_tag["cnt"] / df_tag["cnt"].sum()
df_tag

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\50098728.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tag = pd.read_sql(q, conn, params=[start, end])


,TAG_DESCRIPTION,cnt,share
0,FALLA PI1325,879,0.389110
1,PRESION GAS A QUEMADOR,474,0.209827
2,DETECCION HIDROGENO,225,0.099602
3,DETECCION GAS,108,0.047809
4,DETECCION LLAMA HIDROGENO,84,0.037185
5,FALLA FI1309,78,0.034529
6,DETECCION LLAMA,72,0.031873
7,CABEZA N-701,60,0.026560
8,AVISADOR EMERGENCIA,54,0.023904
9,TEMPERATURA,48,0.021248


In [11]:
q = f"""
SELECT
    PRIORITY,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY PRIORITY
ORDER BY PRIORITY
"""

df_prio = pd.read_sql(q, conn, params=[start, end])
df_prio["share"] = df_prio["cnt"] / df_prio["cnt"].sum()
df_prio

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\1342712363.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_prio = pd.read_sql(q, conn, params=[start, end])


,PRIORITY,cnt,share
0,1,10872,0.873043
1,2,969,0.077813
2,3,612,0.049145


In [12]:
q = f"""
SELECT
    DATEADD(minute, DATEDIFF(minute, 0, ALARMDATETIME), 0) AS minute,
    COUNT(*) AS alarms
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY DATEADD(minute, DATEDIFF(minute, 0, ALARMDATETIME), 0)
ORDER BY minute
"""

df_min = pd.read_sql(q, conn, params=[start, end])
df_min.sort_values("alarms", ascending=False).head(10)

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\2495917982.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_min = pd.read_sql(q, conn, params=[start, end])


,minute,alarms
17,2025-05-03 09:27:00,7734
0,2025-05-03 09:07:00,2877
1,2025-05-03 09:08:00,402
2,2025-05-03 09:09:00,372
18,2025-05-03 09:28:00,291
3,2025-05-03 09:10:00,141
4,2025-05-03 09:11:00,120
16,2025-05-03 09:26:00,117
7,2025-05-03 09:14:00,66
6,2025-05-03 09:13:00,57


In [13]:
start = "2024-06-10 00:00:00"
end   = "2024-06-10 10:33:08"

In [14]:
q = f"""
SELECT TOP 15
    TAG_DESCRIPTION,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY TAG_DESCRIPTION
ORDER BY cnt DESC
"""

df_tag = pd.read_sql(q, conn, params=[start, end])
df_tag["share"] = df_tag["cnt"] / df_tag["cnt"].sum()
df_tag

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\50098728.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tag = pd.read_sql(q, conn, params=[start, end])


,TAG_DESCRIPTION,cnt,share
0,GAS A QUEMADORES L-171,57780,0.987085
1,TRANSMITTER FAIL TT-9706A-1,174,0.002973
2,TRANSMITTER FAIL TT-9706A-2,174,0.002973
3,MANUAL VLV TOGGLE ON/OFF,72,0.001230
4,SWITCH MODE TOGGLE MANUAL/AUTO,66,0.001128
5,FONDO COLUMNA N-1301,48,0.000820
6,VLAVE MODE TOGGLE MANUAL/AUTO,42,0.000718
7,SWITCHOVER STATUS,30,0.000513
8,ESTADO MOTOR DE BOMBA J-1901A,24,0.000410
9,SWITCHOVER MODE,24,0.000410


In [15]:
q = f"""
SELECT TOP 15
    ALARM_DESCRIPTION,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY ALARM_DESCRIPTION
ORDER BY cnt DESC
"""

df_msg = pd.read_sql(q, conn, params=[start, end])
df_msg["share"] = df_msg["cnt"] / df_msg["cnt"].sum()
df_msg

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\1763310739.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_msg = pd.read_sql(q, conn, params=[start, end])


,ALARM_DESCRIPTION,cnt,share
0,NORMAL,29349,0.496750
1,ALARMA,28926,0.489591
2,FALLA,219,0.003707
3,ALTO NIVEL,96,0.001625
4,MANUAL,81,0.001371
5,AUTO,75,0.001269
6,ALTA,60,0.001016
7,BAJA TEMP.,51,0.000863
8,OFF,48,0.000812
9,BAJO NIVEL,42,0.000711


In [16]:
q = f"""
SELECT
    DATEADD(minute, DATEDIFF(minute, 0, ALARMDATETIME), 0) AS minute,
    COUNT(*) AS alarms
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY DATEADD(minute, DATEDIFF(minute, 0, ALARMDATETIME), 0)
ORDER BY minute
"""

df_min = pd.read_sql(q, conn, params=[start, end])
df_min.sort_values("alarms", ascending=False).head(10)

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\2495917982.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_min = pd.read_sql(q, conn, params=[start, end])


,minute,alarms
281,2024-06-10 04:47:00,432
278,2024-06-10 04:44:00,432
275,2024-06-10 04:41:00,432
273,2024-06-10 04:39:00,411
293,2024-06-10 04:59:00,360
290,2024-06-10 04:56:00,360
287,2024-06-10 04:53:00,360
276,2024-06-10 04:42:00,324
282,2024-06-10 04:48:00,324
283,2024-06-10 04:49:00,324


In [17]:
start = "2025-09-05 21:01:15"
end   = "2025-09-05 21:20:53"

In [18]:
q = f"""
SELECT TOP 15
    TAG_DESCRIPTION,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY TAG_DESCRIPTION
ORDER BY cnt DESC
"""

df_tag = pd.read_sql(q, conn, params=[start, end])
df_tag["share"] = df_tag["cnt"] / df_tag["cnt"].sum()
df_tag

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\50098728.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tag = pd.read_sql(q, conn, params=[start, end])


,TAG_DESCRIPTION,cnt,share
0,ALARMA M. BAJA PT_1401,1095,0.325022
1,ALARMA LOW PT_1401,1095,0.325022
2,PRESION FUEL GAS A L1401,1095,0.325022
3,DET LLAMA QUEM 3 L-153B TRIP,60,0.017809
4,DET LLAMA QUEM 1 L-154 TRIP,12,0.003562
5,CAT REGEN TRIP,12,0.003562


In [19]:
q = f"""
SELECT TOP 15
    ALARM_DESCRIPTION,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY ALARM_DESCRIPTION
ORDER BY cnt DESC
"""

df_msg = pd.read_sql(q, conn, params=[start, end])
df_msg["share"] = df_msg["cnt"] / df_msg["cnt"].sum()
df_msg

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\1763310739.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_msg = pd.read_sql(q, conn, params=[start, end])


,ALARM_DESCRIPTION,cnt,share
0,NaN,3285,0.975067
1,NORMAL,42,0.012467
2,TRIP,36,0.010686
3,M.BAJA,6,0.001781


In [20]:
q = f"""
SELECT
    PRIORITY,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY PRIORITY
ORDER BY PRIORITY
"""

df_prio = pd.read_sql(q, conn, params=[start, end])
df_prio["share"] = df_prio["cnt"] / df_prio["cnt"].sum()
df_prio

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\1342712363.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_prio = pd.read_sql(q, conn, params=[start, end])


,PRIORITY,cnt,share
0,1,3357,0.996438
1,3,12,0.003562


In [21]:
start = "2025-06-02 11:16:50"
end   = "2025-06-03 07:09:34"

In [22]:
q = f"""
SELECT TOP 20
    ALARM_DESCRIPTION,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY ALARM_DESCRIPTION
ORDER BY cnt DESC
"""

df_msg = pd.read_sql(q, conn, params=[start, end])
df_msg["share"] = df_msg["cnt"] / df_msg["cnt"].sum()
df_msg

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\668875973.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_msg = pd.read_sql(q, conn, params=[start, end])


,ALARM_DESCRIPTION,cnt,share
0,NaN,36816,0.672438
1,NORMAL,8088,0.147726
2,TRIP,5280,0.096438
3,ALTA PRESION,1140,0.020822
4,BAJO NIVEL,402,0.007342
5,ALTO NIVEL,393,0.007178
6,ALARMA,309,0.005644
7,FALLA,261,0.004767
8,ALTA TEMP.,243,0.004438
9,M.A. PRESION,243,0.004438


In [23]:
q = f"""
SELECT TOP 20
    TAG_DESCRIPTION,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY TAG_DESCRIPTION
ORDER BY cnt DESC
"""

df_tag = pd.read_sql(q, conn, params=[start, end])
df_tag["share"] = df_tag["cnt"] / df_tag["cnt"].sum()
df_tag

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\1886316488.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tag = pd.read_sql(q, conn, params=[start, end])


,TAG_DESCRIPTION,cnt,share
0,DET LLAMA QUEM 4 L-151 TRIP,10185,0.199882
1,ALARMA M. BAJA BI_L1401_1,9327,0.183044
2,ALARMA BAJA BI_L1401_1,9327,0.183044
3,ALARMA LOW FT_1402,9081,0.178216
4,ALARMA M. BAJO FT_1402,9081,0.178216
5,PRESION CONVECCION L-171,2676,0.052517
6,PULSADOR ACEP. DE FALLA L-121,216,0.004239
7,M.B.DELTAP LIFT GAS/REACT PURG,180,0.003533
8,GAS A QUEMADORES L-171,108,0.002120
9,SALIDA REACTOR K-904,87,0.001707


In [24]:
q = f"""
SELECT
    PRIORITY,
    COUNT(*) AS cnt
FROM [{DB}].[{SCHEMA}].[{TABLE}]
WHERE ALARMDATETIME BETWEEN ? AND ?
GROUP BY PRIORITY
ORDER BY PRIORITY
"""

df_prio = pd.read_sql(q, conn, params=[start, end])
df_prio["share"] = df_prio["cnt"] / df_prio["cnt"].sum()
df_prio

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\1342712363.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_prio = pd.read_sql(q, conn, params=[start, end])


,PRIORITY,cnt,share
0,1,52152,0.925666
1,2,1941,0.034452
2,3,2247,0.039883


In [25]:
import pandas as pd

# df es tu blocks_features ya cargado
# te recomiendo empezar con candidatos (para no consultar miles de bloques)
cand = df[df["flood_candidate"] == True].copy()

def fetch_prio1_share(conn, start, end):
    q = f"""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN PRIORITY = 1 THEN 1 ELSE 0 END) AS prio1
    FROM [{DB}].[{SCHEMA}].[{TABLE}]
    WHERE ALARMDATETIME BETWEEN ? AND ?
    """
    r = pd.read_sql(q, conn, params=[start, end]).iloc[0]
    total = int(r["total"])
    prio1 = int(r["prio1"]) if pd.notna(r["prio1"]) else 0
    return (prio1 / total) if total > 0 else 0.0

# ejemplo: solo para los 84 tipificados (rápido)
cand["prio1_share"] = [
    fetch_prio1_share(conn, s, e) for s, e in zip(cand["start"], cand["end"])
]

cand[["start","end","max_rate","flood_severity","flood_type","prio1_share"]].head(10)

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\455496714.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  r = pd.read_sql(q, conn, params=[start, end]).iloc[0]
C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\455496714.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  r = pd.read_sql(q, conn, params=[start, end]).iloc[0]
C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_16916\455496714.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  r = pd.read_sql(q, conn, params=[start, end]).iloc[0]
C:\Users\Andr

,start,end,max_rate,flood_severity,flood_type,prio1_share
167,2024-02-10 20:32:44,2024-02-10 20:51:30,204,medium,OSCILLATION_SINGLE_POINT,0.989899
169,2024-02-10 21:16:10,2024-02-10 21:34:44,208,medium,OSCILLATION_SINGLE_POINT,0.997605
289,2024-03-10 07:33:32,2024-03-10 11:21:45,192,medium,OSCILLATION_SINGLE_POINT,0.955671
562,2024-04-12 14:45:43,2024-04-12 14:55:39,216,medium,OSCILLATION_SINGLE_POINT,1.000000
567,2024-05-10 00:00:00,2024-05-11 01:44:22,288,medium,OSCILLATION_SINGLE_POINT,0.985328
632,2024-05-11 23:48:26,2024-05-12 07:13:28,204,medium,OTHER_OR_NO_FLOOD,0.929829
650,2024-05-12 21:11:13,2024-05-12 23:59:54,234,medium,MESSAGE_DOMINANT,0.933201
651,2024-06-10 00:00:00,2024-06-10 10:33:08,432,severe,OSCILLATION_SINGLE_POINT,0.984154
652,2024-06-10 10:38:55,2024-06-11 01:05:58,315,medium,OSCILLATION_SINGLE_POINT,0.920942
661,2024-06-11 03:38:59,2024-06-11 05:12:44,216,medium,TAG_DOMINANT_REPETITION,0.984893


In [26]:
df = df.merge(
    cand[["start","end","prio1_share"]],
    on=["start","end"],
    how="left"
)
df["prio1_share"] = df["prio1_share"].fillna(0.0)